In [1]:
import os
import pandas as pd
from typing import List, Tuple
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import json
from torch.utils.data import DataLoader
from model import LSTMQNet
import torch
from train_utils import OffPolicyTrainer
import torch.nn as nn
from world_cup_env import WorldCupEnv
from agent import DoubleDQNAgent

font = {'size': 16}

matplotlib.rc('font', **font)

In [2]:
DATASET_PATH = "../../dataset"
INDEX_FIELD = "timestamp"
DATA_FIELD = "num_request"
CPD_CANDIDATE_ROOT = "../../change_point_detection/offline_detection/cpd_candidate"
N_LOOKBACK = 4
N_PREDICT = 2


In [3]:
def get_data_file_list(dataset_path: str) -> List[str]:
    return os.listdir(dataset_path)

In [4]:
def read_dataset(csv_path: str,index_field:str,data_field:str) -> Tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(csv_path)
    return df[index_field].to_numpy(), df[data_field].to_numpy()

In [5]:
def read_candidate_cpds(path: str) -> List[int]:
    candidate_cpds = None
    with open(path, "r") as f:
        candidate_cpds = json.load(f)
    return candidate_cpds

In [6]:
def build_dataset(np_data: np.ndarray, candidate_cpds: List[int]):
    x = []
    y = []
    candidate_cpds = np.array(candidate_cpds, dtype=np.int32)
    np_data = np_data/20000.0
    for idx in range(len(np_data)-N_LOOKBACK-N_PREDICT+2):
        x.append(np_data[idx:idx+N_LOOKBACK+N_PREDICT-1].reshape((-1, 1)))
        is_less = candidate_cpds < idx+N_LOOKBACK+N_PREDICT
        future_candidate_idx = np.sum(is_less)
        if future_candidate_idx < len(candidate_cpds):
            nearest_cpd = candidate_cpds[future_candidate_idx]
            if nearest_cpd > idx + N_LOOKBACK+N_PREDICT+N_PREDICT:
                y.append(0)
            elif nearest_cpd >= idx + N_LOOKBACK+N_PREDICT:
                y.append(1)
            # else:
            #     y.append(2)
        else:
            y.append(0)
    # x = np.array(x)
    # y = np.array(y)
    return x, y

In [7]:
workload_to_skip_list = ["workload_1998-06-13", "workload_1998-06-14", "workload_1998-06-20", "workload_1998-06-21", "workload_1998-06-27", "workload_1998-06-28","workload_1998-07-04"]

In [8]:
import json
data_file_list = get_data_file_list(DATASET_PATH)
state_list = None
action_list = None
env = None
agent = DoubleDQNAgent(2)
trainer = OffPolicyTrainer(env, agent, num_episodes=100, replay_buffer_size=128, batch_size=32, discount_factor=0.9, epsilon_start=0.5, epsilon_end=0.1, epsilon_step=20, learning_rate_start=1e-3, learning_rate_end=1e-4, learning_rate_step=100, tau=0.05)
for file_name in data_file_list:
    workload_name = file_name.split(".")[0]
    if workload_name in workload_to_skip_list:
        continue
    print("read %s" % (file_name))
    np_index, np_data = read_dataset(os.path.join(DATASET_PATH, file_name), INDEX_FIELD, DATA_FIELD)
    np_data = np_data/20000.0
    workload_diff = np.diff(np_data).reshape((-1, 1))
    candidate_cpds = read_candidate_cpds(os.path.join(CPD_CANDIDATE_ROOT, workload_name+".json"))
    env = WorldCupEnv(workload_diff, candidate_cpds, N_LOOKBACK, N_PREDICT)
    trainer.set_env(env)
    _, reward_per_episode = trainer.train()
    state_list, action_list = trainer.eval()
    with open("saved_reward/"+workload_name+".json", "w") as f:
        json.dump(reward_per_episode, f, indent=4)
    agent.save("state_dict/"+workload_name+".pth")
    # break

In [9]:
# import json
# data_file_list = get_data_file_list(DATASET_PATH)
# state_list = None
# action_list = None
# env = None
# workload_diff_total = []
# candidate_cpds_total = []
# for file_name in data_file_list:
#     workload_name = file_name.split(".")[0]
#     if workload_name in workload_to_skip_list:
#         continue
#     print("read %s" % (file_name))
#     np_index, np_data = read_dataset(os.path.join(DATASET_PATH, file_name), INDEX_FIELD, DATA_FIELD)
#     np_data = np_data/20000.0
#     workload_diff = np.diff(np_data).reshape((-1, 1))
#     candidate_cpds = read_candidate_cpds(os.path.join(CPD_CANDIDATE_ROOT, workload_name+".json"))
#     candidate_cpds_total.extend(np.add(np.array(candidate_cpds), len(workload_diff_total)).tolist())
#     workload_diff_total.extend(workload_diff)
# env = WorldCupEnv(workload_diff_total, candidate_cpds_total, N_LOOKBACK, N_PREDICT)
# agent = DoubleDQNAgent(2)
# trainer = OffPolicyTrainer(env, agent, num_episodes=1000, replay_buffer_size=1024, batch_size=128, discount_factor=0.9, epsilon_start=0.5, epsilon_end=0.1, epsilon_step=20, learning_rate_start=1e-3, learning_rate_end=1e-4, learning_rate_step=100, tau=0.05)
# trainer.set_env(env)
# _, reward_per_episode = trainer.train()
# state_list, action_list = trainer.eval()
# with open("saved_reward/"+"all.json", "w") as f:
#     json.dump(reward_per_episode, f, indent=4)
# agent.save("state_dict/"+"all.pth")

read workload_1998-06-10.csv
read workload_1998-06-11.csv
read workload_1998-06-12.csv
read workload_1998-06-15.csv
read workload_1998-06-16.csv
read workload_1998-06-17.csv
read workload_1998-06-18.csv
read workload_1998-06-19.csv
read workload_1998-06-22.csv
read workload_1998-06-23.csv
read workload_1998-06-24.csv
read workload_1998-06-25.csv
read workload_1998-06-26.csv
read workload_1998-06-29.csv
read workload_1998-06-30.csv
read workload_1998-07-03.csv
read workload_1998-07-07.csv
read workload_1998-07-08.csv


/home/andrew/projects/autoscaling-k8s/warm_up_selection/by_rl_window_size_transition/agent.py:58: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1706712676143/work/torch/csrc/utils/tensor_new.cpp:261.)
  state_tensor = torch.tensor(state, dtype=torch.float32, device=self.device)


episode 0, loss 0.098392, num_attempts 8410, reward -2515.092262
episode 1, loss 0.133950, num_attempts 8410, reward -2397.419945
episode 2, loss 0.121653, num_attempts 8410, reward -2397.667016
episode 3, loss 0.118190, num_attempts 8410, reward -2366.753151
episode 4, loss 0.077554, num_attempts 8410, reward -2272.636562
episode 5, loss 0.127877, num_attempts 8410, reward -2284.473971
episode 6, loss 0.108572, num_attempts 8410, reward -2233.963012
episode 7, loss 0.105423, num_attempts 8410, reward -2116.515860
episode 8, loss 0.101539, num_attempts 8410, reward -2196.386266
episode 9, loss 0.089247, num_attempts 8410, reward -2008.240559
episode 10, loss 0.100021, num_attempts 8410, reward -1995.279872
episode 11, loss 0.101834, num_attempts 8410, reward -2009.972334
episode 12, loss 0.091293, num_attempts 8410, reward -2062.908770
episode 13, loss 0.117218, num_attempts 8410, reward -1933.202021
episode 14, loss 0.093771, num_attempts 8410, reward -1888.885712
episode 15, loss 0.1

KeyboardInterrupt: 